# Homography correctness desc stats

In [15]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import itertools
import scipy.stats as stats
from IPython.display import display
# ============================================================
# Display options
# ============================================================
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

# ============================================================
# Evaluation directories
# ============================================================
eval_log_dirs = [
    "/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_v16.2-chroma-ISPDefaultInitialHype_original_HPatchesV4.1",
    "/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_sunlit_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_adjust_defaultcolorhuesat_120000_HPatchesV4.1",
    "/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_PRETRAINED_SAMEMODELHOMOADAPT_AGGRESSIVEHOMOADAPT_CFANORMALIZE_XHOMOWARP_DETERMHOMOADAPT_train_v16.2-chroma-HumanTunedInitialHype_sunlit_lr0.005_bothLoss_initialHypeHomoAdaptOnly_gradac187_78500_HPatchesV4.1",
    # "/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_CMAES_train_maxstd0.01_csadampfac10.0_2800_HPatchesV4.1"
    "/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_CMAES_train_maxstd0.01_csadampfac10.0_2290_HPatchesV4.1"
]

# eval_log_dirs = [
#     "/home/boat/proxyISP/pytorch-superpoint/logs/eval_ll_v16.2-chroma-ISPDefaultInitialHype_original_HPatchesV4.1",
#     "/home/boat/proxyISP/pytorch-superpoint/logs/eval_ll_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_lowlight_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_adjust_defaultcolorhuesat_denoise_45000_HPatchesV4.1",
#     "/home/boat/proxyISP/pytorch-superpoint/logs/eval_ll_PRETRAINED_SAMEMODELHOMOADAPT_AGGRESSIVEHOMOADAPT_CFANORMALIZE_XHOMOWARP_DETERMHOMOADAPT_train_v16.2-chroma-HumanTunedInitialHype_lowlight_lr0.005_bothLoss_initialHypeHomoAdaptOnly_gradac187_90200_HPatchesV4.1",
#     "/home/boat/proxyISP/pytorch-superpoint/logs/eval_ll_CMAES_train_lowlight_maxstd0.01_csadampfac10.0_2800_HPatchesV4.1"
# ]

suffix = "sl"

print("=" * 80)
print("eval_log_dirs", "\n", "\n".join(eval_log_dirs))
print("=" * 80)

names = [
    "Orig-ISP",
    "VO-ISP",
    "FO-ISP (Proxy)",
    "FO-ISP (CMA-ES)"
]

# ============================================================
# Load result.npz files
# ============================================================
objects = {}

for k,d in enumerate(eval_log_dirs):

    name = names[k]

    result_path = Path(d) / "predictions" / "result.npz"

    if not result_path.exists():
        raise FileNotFoundError(result_path)

    objects[name] = np.load(result_path)

print("=" * 80)
print("Loaded evaluations")
print("=" * 80)

for k in objects:
    print(k)

# ============================================================
# Print descriptive statistics
# ============================================================
first_name = next(iter(objects))
thresholds = objects[first_name]["homography_thresh"]

print("\n" + "=" * 80)
print(f"Homography Correctness Descriptive Statistics suffix:{suffix}")
print("=" * 80)

all_tables = []

for k_idx, threshold in enumerate(thresholds):

    print(f"\nThreshold @ {threshold}")
    # print("-" * 80)

    stats_df = pd.DataFrame()

    for eval_name, obj in objects.items():

        # ----------------------------------------------------
        # correctness shape:
        # [num_pairs, num_thresholds]
        # ----------------------------------------------------
        correctness = obj["correctness"][:, k_idx]

        # bool -> float
        correctness = correctness.astype(np.float32)

        # ----------------------------------------------------
        # descriptive statistics
        # ----------------------------------------------------
        mean_ = correctness.mean()
        std_ = correctness.std()
        median_ = np.median(correctness)
        min_ = correctness.min()
        max_ = correctness.max()

        success_count = int(correctness.sum())
        total_count = len(correctness)

        success_rate = mean_ * 100

        # ----------------------------------------------------
        # save table row
        # ----------------------------------------------------
        stats_df.at[eval_name, "mean"] = mean_
        stats_df.at[eval_name, "std"] = std_
        stats_df.at[eval_name, "median"] = median_
        stats_df.at[eval_name, "min"] = min_
        stats_df.at[eval_name, "max"] = max_
        stats_df.at[eval_name, "success_count"] = success_count
        stats_df.at[eval_name, "total_pairs"] = total_count
        stats_df.at[eval_name, "success_rate_%"] = success_rate

        # # ----------------------------------------------------
        # # pretty print
        # # ----------------------------------------------------
        # print(
        #     f"{eval_name}"
        #     f"\n    mean           : {mean_:.4f}"
        #     f"\n    std            : {std_:.4f}"
        #     f"\n    median         : {median_:.4f}"
        #     f"\n    min            : {min_:.4f}"
        #     f"\n    max            : {max_:.4f}"
        #     f"\n    success_count  : {success_count}"
        #     f"\n    total_pairs    : {total_count}"
        #     f"\n    success_rate   : {success_rate:.2f}%"
        #     "\n"
        # )

    stats_df = stats_df.round(4)

    all_tables.append((threshold, stats_df))

    display(stats_df)

# ============================================================
# Optional: save all stats to CSV
# ============================================================
save_csv = False

if save_csv:

    save_dir = Path("homography_stats")
    save_dir.mkdir(exist_ok=True)

    for threshold, df in all_tables:

        csv_path = save_dir / f"homography_stats_threshold_{threshold}.csv"

        df.to_csv(csv_path)

        print(f"Saved: {csv_path}")

eval_log_dirs 
 /home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_v16.2-chroma-ISPDefaultInitialHype_original_HPatchesV4.1
/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_sunlit_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_adjust_defaultcolorhuesat_120000_HPatchesV4.1
/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_PRETRAINED_SAMEMODELHOMOADAPT_AGGRESSIVEHOMOADAPT_CFANORMALIZE_XHOMOWARP_DETERMHOMOADAPT_train_v16.2-chroma-HumanTunedInitialHype_sunlit_lr0.005_bothLoss_initialHypeHomoAdaptOnly_gradac187_78500_HPatchesV4.1
/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_CMAES_train_maxstd0.01_csadampfac10.0_2290_HPatchesV4.1
Loaded evaluations
Orig-ISP
VO-ISP
FO-ISP (Proxy)
FO-ISP (CMA-ES)

Homography Correctness Descriptive Statistics suffix:sl

Threshold @ 1


,mean,std,median,min,max,success_count,total_pairs,success_rate_%
Orig-ISP,0.2786,0.4483,0.0,0.0,1.0,39.0,140.0,27.8571
VO-ISP,0.2929,0.4551,0.0,0.0,1.0,41.0,140.0,29.2857
FO-ISP (Proxy),0.3071,0.4613,0.0,0.0,1.0,43.0,140.0,30.7143
FO-ISP (CMA-ES),0.2857,0.4518,0.0,0.0,1.0,40.0,140.0,28.5714



Threshold @ 3


,mean,std,median,min,max,success_count,total_pairs,success_rate_%
Orig-ISP,0.5071,0.4999,1.0,0.0,1.0,71.0,140.0,50.7143
VO-ISP,0.5429,0.4982,1.0,0.0,1.0,76.0,140.0,54.2857
FO-ISP (Proxy),0.5500,0.4975,1.0,0.0,1.0,77.0,140.0,55.0000
FO-ISP (CMA-ES),0.5071,0.4999,1.0,0.0,1.0,71.0,140.0,50.7143



Threshold @ 5


...

,mean,std,median,min,max,success_count,total_pairs,success_rate_%
Orig-ISP,0.7500,0.4330,1.0,0.0,1.0,105.0,140.0,75.0000
VO-ISP,0.7571,0.4288,1.0,0.0,1.0,106.0,140.0,75.7143
FO-ISP (Proxy),0.7929,0.4053,1.0,0.0,1.0,111.0,140.0,79.2857
FO-ISP (CMA-ES),0.7571,0.4288,1.0,0.0,1.0,106.0,140.0,75.7143



Threshold @ 20


,mean,std,median,min,max,success_count,total_pairs,success_rate_%
Orig-ISP,0.8071,0.3945,1.0,0.0,1.0,113.0,140.0,80.7143
VO-ISP,0.8071,0.3945,1.0,0.0,1.0,113.0,140.0,80.7143
FO-ISP (Proxy),0.8214,0.3830,1.0,0.0,1.0,115.0,140.0,82.1429
FO-ISP (CMA-ES),0.8429,0.3639,1.0,0.0,1.0,118.0,140.0,84.2857



Threshold @ 50


,mean,std,median,min,max,success_count,total_pairs,success_rate_%
Orig-ISP,0.8571,0.3499,1.0,0.0,1.0,120.0,140.0,85.7143
VO-ISP,0.8714,0.3347,1.0,0.0,1.0,122.0,140.0,87.1429
FO-ISP (Proxy),0.9000,0.3000,1.0,0.0,1.0,126.0,140.0,90.0000
FO-ISP (CMA-ES),0.8857,0.3182,1.0,0.0,1.0,124.0,140.0,88.5714


# Homography Correctness hyp test

In [16]:
import itertools
import numpy as np
import pandas as pd
from scipy import stats


# ============================================================
# McNemar test helper
# ============================================================
def mcnemar_test(x, y):
    """
    x, y: binary arrays (0/1)
    """

    x = x.astype(bool)
    y = y.astype(bool)

    # contingency
    b = np.sum((x == 1) & (y == 0))  # A correct, B wrong
    c = np.sum((x == 0) & (y == 1))  # A wrong, B correct

    if (b + c) == 0:
        return b, c, 0.0, 1.0

    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    p = 1 - stats.chi2.cdf(chi2, df=1)

    return b, c, chi2, p


# ============================================================
# Hypothesis testing (McNemar only)
# ============================================================

first = next(iter(objects))
thresholds = objects[first]["homography_thresh"]
keys = list(objects.keys())

all_tables = {}

print("\n" + "=" * 80)
print(f"Homography Correctness McNemar Test: {suffix}")
print("=" * 80)

for k_idx, thr in enumerate(thresholds):

    rows = []

    for a, b in itertools.combinations(keys, 2):

        A = objects[a]["correctness"][:, k_idx].astype(np.int32)
        B = objects[b]["correctness"][:, k_idx].astype(np.int32)

        b_count, c_count, chi2, p = mcnemar_test(A, B)

        rows.append({
            "model_A": a,
            "model_B": b,
            "acc_A": A.mean(),
            "acc_B": B.mean(),
            "diff": A.mean() - B.mean(),
            "A_correct_B_wrong": b_count,
            "A_wrong_B_correct": c_count,
            # "chi2": chi2,
            "p_value": p,
            "significant (p < 0.05)": p < 0.05,
        })

    df = pd.DataFrame(rows)

    # df = df.sort_values(
    #     ["significant (p < 0.05)", "p_value"],
    #     ascending=[False, True]
    # ).reset_index(drop=True)

    all_tables[thr] = df

    print(f"\nThreshold @ {thr}")
    display(df)
    # df
    # print(df.to_string(
    #     index=False,
    #     float_format=lambda x: f"{x:.5f}"
    # ))


Homography Correctness McNemar Test: sl

Threshold @ 1


,model_A,model_B,acc_A,acc_B,diff,A_correct_B_wrong,A_wrong_B_correct,p_value,significant (p < 0.05)
0,Orig-ISP,VO-ISP,0.278571,0.292857,-0.014286,8,10,0.813664,False
1,Orig-ISP,FO-ISP (Proxy),0.278571,0.307143,-0.028571,7,11,0.479500,False
2,Orig-ISP,FO-ISP (CMA-ES),0.278571,0.285714,-0.007143,11,12,1.000000,False
3,VO-ISP,FO-ISP (Proxy),0.292857,0.307143,-0.014286,7,9,0.802587,False
4,VO-ISP,FO-ISP (CMA-ES),0.292857,0.285714,0.007143,8,7,1.000000,False
5,FO-ISP (Proxy),FO-ISP (CMA-ES),0.307143,0.285714,0.021429,10,7,0.627626,False



Threshold @ 3


,model_A,model_B,acc_A,acc_B,diff,A_correct_B_wrong,A_wrong_B_correct,p_value,significant (p < 0.05)
0,Orig-ISP,VO-ISP,0.507143,0.542857,-0.035714,10,15,0.423711,False
1,Orig-ISP,FO-ISP (Proxy),0.507143,0.550000,-0.042857,11,17,0.344704,False
2,Orig-ISP,FO-ISP (CMA-ES),0.507143,0.507143,0.000000,11,11,0.831170,False
3,VO-ISP,FO-ISP (Proxy),0.542857,0.550000,-0.007143,13,14,1.000000,False
4,VO-ISP,FO-ISP (CMA-ES),0.542857,0.507143,0.035714,11,6,0.331975,False
5,FO-ISP (Proxy),FO-ISP (CMA-ES),0.550000,0.507143,0.042857,17,11,0.344704,False



Threshold @ 5


...

,model_A,model_B,acc_A,acc_B,diff,A_correct_B_wrong,A_wrong_B_correct,p_value,significant (p < 0.05)
0,Orig-ISP,VO-ISP,0.750000,0.757143,-0.007143,4,5,1.000000,False
1,Orig-ISP,FO-ISP (Proxy),0.750000,0.792857,-0.042857,2,8,0.113846,False
2,Orig-ISP,FO-ISP (CMA-ES),0.750000,0.757143,-0.007143,7,8,1.000000,False
3,VO-ISP,FO-ISP (Proxy),0.757143,0.792857,-0.035714,2,7,0.182422,False
4,VO-ISP,FO-ISP (CMA-ES),0.757143,0.757143,0.000000,6,6,0.772830,False
5,FO-ISP (Proxy),FO-ISP (CMA-ES),0.792857,0.757143,0.035714,9,4,0.267257,False



Threshold @ 20


,model_A,model_B,acc_A,acc_B,diff,A_correct_B_wrong,A_wrong_B_correct,p_value,significant (p < 0.05)
0,Orig-ISP,VO-ISP,0.807143,0.807143,0.000000,3,3,0.683091,False
1,Orig-ISP,FO-ISP (Proxy),0.807143,0.821429,-0.014286,3,5,0.723674,False
2,Orig-ISP,FO-ISP (CMA-ES),0.807143,0.842857,-0.035714,2,7,0.182422,False
3,VO-ISP,FO-ISP (Proxy),0.807143,0.821429,-0.014286,3,5,0.723674,False
4,VO-ISP,FO-ISP (CMA-ES),0.807143,0.842857,-0.035714,2,7,0.182422,False
5,FO-ISP (Proxy),FO-ISP (CMA-ES),0.821429,0.842857,-0.021429,4,7,0.546494,False



Threshold @ 50


,model_A,model_B,acc_A,acc_B,diff,A_correct_B_wrong,A_wrong_B_correct,p_value,significant (p < 0.05)
0,Orig-ISP,VO-ISP,0.857143,0.871429,-0.014286,1,3,0.617075,False
1,Orig-ISP,FO-ISP (Proxy),0.857143,0.900000,-0.042857,1,7,0.077100,False
2,Orig-ISP,FO-ISP (CMA-ES),0.857143,0.885714,-0.028571,2,6,0.288844,False
3,VO-ISP,FO-ISP (Proxy),0.871429,0.900000,-0.028571,0,4,0.133614,False
4,VO-ISP,FO-ISP (CMA-ES),0.871429,0.885714,-0.014286,2,4,0.683091,False
5,FO-ISP (Proxy),FO-ISP (CMA-ES),0.900000,0.885714,0.014286,4,2,0.683091,False


# Loc Err desc stats

In [17]:
# ============================================================
# Generic descriptive statistics utility
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Helper: stats function
# ============================================================
def compute_stats(x):
    x = np.asarray(x).reshape(-1)

    return {
        "mean": np.mean(x),
        "std": np.std(x),
        "median": np.median(x),
        "min": np.min(x),
        "max": np.max(x),
        "rmse": np.sqrt(np.mean(x**2)),
        "p95": np.percentile(x, 95),
        "p90": np.percentile(x, 90),
    }


# ============================================================
# Main function
# ============================================================
def summarize_metric(objects, key, suffix="", verbose = True):
    """
    Compute descriptive statistics for a given key inside objects.

    Parameters
    ----------
    objects : dict
        Dictionary of experiment/object results.

    key : str
        Metric key to analyze.
        Example:
            "localization_err"
            "rotation_err"
            "confidence"

    suffix : str
        Optional suffix for printing.
    """

    print("\n" + "=" * 80)
    print(f"{key} Descriptive Statistics {suffix}")
    print("=" * 80)

    all_results = {}

    for name, obj in objects.items():

        if key not in obj:
            raise KeyError(
                f"{key} not found in {name}. "
                f"Available keys: {list(obj.keys())}"
            )

        values = obj[key]

        stats = compute_stats(values)

        all_results[name] = stats
        
        if verbose:
            print(f"\n{name}")
            print("-" * 80)

            for k, v in stats.items():
                print(f"{k:10s}: {v:.6f}")

    # ========================================================
    # Summary table
    # ========================================================
    df = pd.DataFrame(all_results).T
    df = df.round(6)

    # print("\n" + "=" * 80)
    # print("Summary Table")
    # print("=" * 80)

    display(df)

    return df


# ============================================================
# Example usage
# ============================================================

# localization error
df_loc = summarize_metric(
    objects,
    key="localization_err",
    suffix="(Localization)" + " subset: " + suffix,
    verbose=False
)


localization_err Descriptive Statistics (Localization) subset: sl


,mean,std,median,min,max,rmse,p95,p90
Orig-ISP,1.023654,0.252914,1.007145,0.447708,1.583136,1.054435,1.421770,1.371913
VO-ISP,1.023248,0.252458,1.018326,0.470630,1.554211,1.053931,1.441579,1.357078
FO-ISP (Proxy),1.015121,0.259218,1.000132,0.486988,1.550300,1.047695,1.426030,1.370580
FO-ISP (CMA-ES),1.015934,0.254466,1.005804,0.460023,1.572332,1.047318,1.417189,1.369130


# Loc Err hyp test

In [18]:
import itertools
import numpy as np
import pandas as pd
import scipy.stats as stats


# ============================================================
# Statistical test helpers
# ============================================================
def wilcoxon_test(x, y):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    try:
        stat, p = stats.wilcoxon(x, y, alternative="two-sided")
    except ValueError:
        # occurs if all paired differences are zero
        return 0.0, 1.0

    return stat, p


def paired_t_test(x, y):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    stat, p = stats.ttest_rel(x, y)

    return stat, p


# ============================================================
# Main hypothesis testing function
# ============================================================
def hypothesis_testing_ttest_wilcox(objects, key, suffix="", alpha=0.05, verbose = True):
    """
    Perform pairwise statistical hypothesis testing.

    Parameters
    ----------
    objects : dict
        Dictionary containing experiment results.

    key : str
        Metric key to compare.
        Example:
            "localization_err"
            "rotation_err"

    suffix : str
        Optional title suffix.

    alpha : float
        Significance threshold.
    """

    print("\n" + "=" * 80)
    print(f"{key} Hypothesis Testing {suffix}")
    print("=" * 80)

    keys = list(objects.keys())

    summary_rows = []

    for a, b in itertools.combinations(keys, 2):

        if key not in objects[a]:
            raise KeyError(f"{key} not found in {a}")

        if key not in objects[b]:
            raise KeyError(f"{key} not found in {b}")

        A = np.asarray(objects[a][key]).reshape(-1)
        B = np.asarray(objects[b][key]).reshape(-1)

        # ----------------------------------------------------
        # paired assumption
        # ----------------------------------------------------
        if len(A) != len(B):
            raise ValueError(
                f"Mismatch length: {a}={len(A)}, {b}={len(B)}"
            )

        # ----------------------------------------------------
        # effect size
        # ----------------------------------------------------
        mean_diff = A.mean() - B.mean()

        # ----------------------------------------------------
        # Wilcoxon
        # ----------------------------------------------------
        w_stat, w_p = wilcoxon_test(A, B)

        # ----------------------------------------------------
        # Paired t-test
        # ----------------------------------------------------
        t_stat, t_p = paired_t_test(A, B)

        # ----------------------------------------------------
        # significance
        # ----------------------------------------------------
        significant = w_p < alpha

        # ----------------------------------------------------
        # print
        # ----------------------------------------------------
        if verbose:
            print(f"\n{a} vs {b}")
            print("-" * 60)
    
            print(f"Mean(A) - Mean(B): {mean_diff:.6f}")
    
            print("\nWilcoxon signed-rank test")
            print(f"  statistic : {w_stat:.6f}")
            print(f"  p-value   : {w_p:.6e}")
    
            print("\nPaired t-test")
            print(f"  t-stat    : {t_stat:.6f}")
            print(f"  p-value   : {t_p:.6e}")
    
            if significant:
                print("\nResult: SIGNIFICANT difference (Wilcoxon)")
            else:
                print("\nResult: NOT significant (Wilcoxon)")

        # ----------------------------------------------------
        # summary row
        # ----------------------------------------------------
        summary_rows.append({
            "A": a,
            "B": b,
            "mean A": A.mean(),
            "mean B": B.mean(),
            "mean_diff": mean_diff,
            "wilcoxon_stat": w_stat,
            "wilcoxon_p": w_p,
            "t_stat": t_stat,
            "t_p": t_p,
            "significant": significant,
        })

    # ========================================================
    # summary dataframe
    # ========================================================
    df = pd.DataFrame(summary_rows)

    # print("\n" + "=" * 80)
    # print("Summary Table")
    # print("=" * 80)

    display(df)

    return df


# ============================================================
# Example usage
# ============================================================

df_loc = hypothesis_testing_ttest_wilcox(
    objects,
    key="localization_err",
    suffix="(Localization)"  + " subset: " + suffix,
    verbose=False
)


localization_err Hypothesis Testing (Localization) subset: sl


,A,B,mean A,mean B,mean_diff,wilcoxon_stat,wilcoxon_p,t_stat,t_p,significant
0,Orig-ISP,VO-ISP,1.023654,1.023248,0.000407,4921.0,0.976768,0.100477,0.920110,False
1,Orig-ISP,FO-ISP (Proxy),1.023654,1.015121,0.008533,3541.0,0.003736,2.486408,0.014088,True
2,Orig-ISP,FO-ISP (CMA-ES),1.023654,1.015934,0.007720,4134.0,0.095686,1.893386,0.060385,False
3,VO-ISP,FO-ISP (Proxy),1.023248,1.015121,0.008127,3410.0,0.001513,2.475933,0.014490,True
4,VO-ISP,FO-ISP (CMA-ES),1.023248,1.015934,0.007314,3698.0,0.010081,2.882071,0.004579,True
5,FO-ISP (Proxy),FO-ISP (CMA-ES),1.015121,1.015934,-0.000813,4507.0,0.373320,-0.248670,0.803983,False


# Repeatability

In [19]:
df_loc = summarize_metric(
    objects,
    key="repeatability",
    suffix="(repeatability)"  + " subset: " + suffix,
    verbose=False
)


repeatability Descriptive Statistics (repeatability) subset: sl


,mean,std,median,min,max,rmse,p95,p90
Orig-ISP,0.656290,0.128575,0.680275,0.281818,0.856712,0.668766,0.831342,0.802168
VO-ISP,0.657822,0.135882,0.685528,0.273171,0.888889,0.671710,0.844354,0.816779
FO-ISP (Proxy),0.658009,0.135127,0.680156,0.278826,0.888399,0.671740,0.849167,0.815724
FO-ISP (CMA-ES),0.660041,0.135241,0.676861,0.253191,0.901361,0.673754,0.846977,0.827336


# Repeatability hyp test

In [20]:
df_loc = hypothesis_testing_ttest_wilcox(
    objects,
    key="repeatability",
    suffix="(Repeatability)"  + " subset: " + suffix,
    verbose = False
)


repeatability Hypothesis Testing (Repeatability) subset: sl


,A,B,mean A,mean B,mean_diff,wilcoxon_stat,wilcoxon_p,t_stat,t_p,significant
0,Orig-ISP,VO-ISP,0.656290,0.657822,-0.001532,4528.0,0.397224,-0.700535,0.484764,False
1,Orig-ISP,FO-ISP (Proxy),0.656290,0.658009,-0.001718,4736.0,0.678922,-0.794020,0.428538,False
2,Orig-ISP,FO-ISP (CMA-ES),0.656290,0.660041,-0.003751,4182.0,0.117279,-1.544893,0.124646,False
3,VO-ISP,FO-ISP (Proxy),0.657822,0.658009,-0.000186,4858.0,0.872751,-0.147264,0.883137,False
4,VO-ISP,FO-ISP (CMA-ES),0.657822,0.660041,-0.002219,4228.0,0.180479,-1.811225,0.072265,False
5,FO-ISP (Proxy),FO-ISP (CMA-ES),0.658009,0.660041,-0.002032,4452.0,0.385217,-1.369653,0.173004,False


# mAP

In [21]:
df_loc = summarize_metric(
    objects,
    key="mAP",
    suffix="(mAP)"  + " subset: " + suffix,
    verbose = False
)


mAP Descriptive Statistics (mAP) subset: sl


,mean,std,median,min,max,rmse,p95,p90
Orig-ISP,0.756796,0.234114,0.841226,0.077020,0.992295,0.792180,0.976794,0.970057
VO-ISP,0.765682,0.224121,0.852807,0.164107,0.991673,0.797809,0.982001,0.970730
FO-ISP (Proxy),0.771599,0.222658,0.857848,0.069529,0.991311,0.803082,0.980922,0.971235
FO-ISP (CMA-ES),0.767613,0.234327,0.860556,0.055091,0.995159,0.802583,0.984597,0.975635


In [22]:
df_loc = hypothesis_testing_ttest_wilcox(
    objects,
    key="mAP",
    suffix="(mAP)"  + " subset: " + suffix,
    verbose = False
)


mAP Hypothesis Testing (mAP) subset: sl


,A,B,mean A,mean B,mean_diff,wilcoxon_stat,wilcoxon_p,t_stat,t_p,significant
0,Orig-ISP,VO-ISP,0.756796,0.765682,-0.008886,4432.0,0.295433,-1.375291,0.171254,False
1,Orig-ISP,FO-ISP (Proxy),0.756796,0.771599,-0.014803,3589.0,0.005114,-2.568079,0.011282,True
2,Orig-ISP,FO-ISP (CMA-ES),0.756796,0.767613,-0.010817,3656.0,0.007804,-1.790347,0.075575,True
3,VO-ISP,FO-ISP (Proxy),0.765682,0.771599,-0.005917,4695.0,0.617626,-1.040851,0.299752,False
4,VO-ISP,FO-ISP (CMA-ES),0.765682,0.767613,-0.001931,4182.0,0.117279,-0.309880,0.757116,False
5,FO-ISP (Proxy),FO-ISP (CMA-ES),0.771599,0.767613,0.003986,4558.0,0.432930,0.626125,0.532260,False


# Match Score

In [23]:
df_loc = summarize_metric(
    objects,
    key="mscore",
    suffix="(mscore)" + " subset: " + suffix,
    verbose = False
)


mscore Descriptive Statistics (mscore) subset: sl


,mean,std,median,min,max,rmse,p95,p90
Orig-ISP,0.343413,0.237462,0.298348,0.018919,0.835165,0.417517,0.742146,0.703199
VO-ISP,0.347015,0.239513,0.315018,0.019048,0.842222,0.421646,0.765948,0.719833
FO-ISP (Proxy),0.352831,0.240180,0.327232,0.016043,0.805333,0.426821,0.765996,0.719243
FO-ISP (CMA-ES),0.348291,0.239243,0.312303,0.016949,0.819974,0.422545,0.759274,0.739723


In [24]:
df_loc = hypothesis_testing_ttest_wilcox(
    objects,
    key="mscore",
    suffix="(mscore)"  + " subset: " + suffix,
    verbose = False
)


mscore Hypothesis Testing (mscore) subset: sl


,A,B,mean A,mean B,mean_diff,wilcoxon_stat,wilcoxon_p,t_stat,t_p,significant
0,Orig-ISP,VO-ISP,0.343413,0.347015,-0.003602,4539.0,0.410105,-1.342491,0.181625,False
1,Orig-ISP,FO-ISP (Proxy),0.343413,0.352831,-0.009419,3542.0,0.003761,-2.935303,0.003900,True
2,Orig-ISP,FO-ISP (CMA-ES),0.343413,0.348291,-0.004879,3997.0,0.051044,-1.666721,0.097822,False
3,VO-ISP,FO-ISP (Proxy),0.347015,0.352831,-0.005817,3711.0,0.010896,-2.177473,0.031133,True
4,VO-ISP,FO-ISP (CMA-ES),0.347015,0.348291,-0.001277,4518.0,0.465659,-0.500353,0.617618,False
5,FO-ISP (Proxy),FO-ISP (CMA-ES),0.352831,0.348291,0.004540,4310.0,0.193585,1.820188,0.070882,False


In [200]:
%%bash -s "$suffix"
jupyter nbconvert --to script Analyze_stats.ipynb --stdout | python > desc_stats_hyp_test_superpoint_$1.txt

[NbConvertApp] Converting notebook Analyze_stats.ipynb to script
Traceback (most recent call last):
  File "<stdin>", line 624, in <module>
NameError: name 'get_ipython' is not defined


CalledProcessError: Command 'b'jupyter nbconvert --to script Analyze_stats.ipynb --stdout | python > desc_stats_hyp_test_superpoint_$1.txt\n'' returned non-zero exit status 1.

# Homography Correctness Desc stats by viewpoint

In [25]:
import numpy as np
import pandas as pd

print("Homography Correctness Desc stats by viewpoint", suffix)
thresholds = [1, 3, 5, 10, 20, 50]

for vp in range(5):

    rows = []

    for method_name, method_obj in objects.items():

        correctness = method_obj["correctness"]

        n_sequences = correctness.shape[0] // 5

        # (sequence, viewpoint_level, threshold)
        corr = correctness.reshape(
            n_sequences,
            5,
            len(thresholds)
        )

        vp_data = corr[:, vp, :].astype(float)

        row = {
            "Method": method_name
        }

        for t_idx, thr in enumerate(thresholds):

            x = vp_data[:, t_idx]

            mean = x.mean() * 100
            std = x.std(ddof=1) * 100

            row[f"@{thr}"] = f"{mean:.2f} ± {std:.2f}"

        rows.append(row)

    table = pd.DataFrame(rows)

    print("=" * 80)
    print(f"Viewpoint Extremeness {vp + 1}")
    print("=" * 80)

    display(table)

Homography Correctness Desc stats by viewpoint sl
Viewpoint Extremeness 1


,Method,@1,@3,@5,@10,@20,@50
0,Orig-ISP,75.00 ± 44.10,96.43 ± 18.90,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00
1,VO-ISP,89.29 ± 31.50,96.43 ± 18.90,96.43 ± 18.90,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00
2,FO-ISP (Proxy),92.86 ± 26.23,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00
3,FO-ISP (CMA-ES),92.86 ± 26.23,96.43 ± 18.90,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00


Viewpoint Extremeness 2


,Method,@1,@3,@5,@10,@20,@50
0,Orig-ISP,50.00 ± 50.92,89.29 ± 31.50,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00
1,VO-ISP,50.00 ± 50.92,92.86 ± 26.23,96.43 ± 18.90,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00
2,FO-ISP (Proxy),39.29 ± 49.73,92.86 ± 26.23,96.43 ± 18.90,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00
3,FO-ISP (CMA-ES),39.29 ± 49.73,92.86 ± 26.23,96.43 ± 18.90,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00


Viewpoint Extremeness 3


,Method,@1,@3,@5,@10,@20,@50
0,Orig-ISP,14.29 ± 35.63,53.57 ± 50.79,78.57 ± 41.79,96.43 ± 18.90,100.00 ± 0.00,100.00 ± 0.00
1,VO-ISP,7.14 ± 26.23,60.71 ± 49.73,75.00 ± 44.10,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00
2,FO-ISP (Proxy),17.86 ± 39.00,50.00 ± 50.92,82.14 ± 39.00,100.00 ± 0.00,100.00 ± 0.00,100.00 ± 0.00
3,FO-ISP (CMA-ES),10.71 ± 31.50,50.00 ± 50.92,85.71 ± 35.63,96.43 ± 18.90,100.00 ± 0.00,100.00 ± 0.00


Viewpoint Extremeness 4


,Method,@1,@3,@5,@10,@20,@50
0,Orig-ISP,0.00 ± 0.00,14.29 ± 35.63,28.57 ± 46.00,67.86 ± 47.56,89.29 ± 31.50,96.43 ± 18.90
1,VO-ISP,0.00 ± 0.00,17.86 ± 39.00,46.43 ± 50.79,75.00 ± 44.10,82.14 ± 39.00,100.00 ± 0.00
2,FO-ISP (Proxy),0.00 ± 0.00,28.57 ± 46.00,46.43 ± 50.79,82.14 ± 39.00,85.71 ± 35.63,100.00 ± 0.00
3,FO-ISP (CMA-ES),0.00 ± 0.00,14.29 ± 35.63,35.71 ± 48.80,67.86 ± 47.56,96.43 ± 18.90,100.00 ± 0.00


Viewpoint Extremeness 5


,Method,@1,@3,@5,@10,@20,@50
0,Orig-ISP,0.00 ± 0.00,0.00 ± 0.00,3.57 ± 18.90,10.71 ± 31.50,14.29 ± 35.63,32.14 ± 47.56
1,VO-ISP,0.00 ± 0.00,3.57 ± 18.90,3.57 ± 18.90,3.57 ± 18.90,21.43 ± 41.79,35.71 ± 48.80
2,FO-ISP (Proxy),3.57 ± 18.90,3.57 ± 18.90,7.14 ± 26.23,14.29 ± 35.63,25.00 ± 44.10,50.00 ± 50.92
3,FO-ISP (CMA-ES),0.00 ± 0.00,0.00 ± 0.00,10.71 ± 31.50,14.29 ± 35.63,25.00 ± 44.10,42.86 ± 50.40


# Homography Correctness Mcnemar test by viewpoint

In [26]:
import numpy as np
import pandas as pd

from itertools import combinations
from statsmodels.stats.contingency_tables import mcnemar

thresholds = [1, 3, 5, 10, 20, 50]

print("Homography Correctness Mcnemar test by viewpoint", suffix)

def p_to_str(p):
    if p < 0.001:
        return f"{p:.3g}***"
    elif p < 0.01:
        return f"{p:.3g}**"
    elif p < 0.05:
        return f"{p:.3g}*"
    else:
        return f"{p:.3g}"


method_names = list(objects.keys())

for vp in range(5):

    rows = []

    for method_a, method_b in combinations(method_names, 2):

        corr_a = objects[method_a]["correctness"]
        corr_b = objects[method_b]["correctness"]

        n_sequences = corr_a.shape[0] // 5

        corr_a = corr_a.reshape(
            n_sequences,
            5,
            len(thresholds)
        )

        corr_b = corr_b.reshape(
            n_sequences,
            5,
            len(thresholds)
        )

        row = {
            "Comparison": f"{method_a} vs {method_b}"
        }

        for t_idx, thr in enumerate(thresholds):

            x = corr_a[:, vp, t_idx].astype(bool)
            y = corr_b[:, vp, t_idx].astype(bool)

            both_correct = np.sum(x & y)
            a_only = np.sum(x & ~y)
            b_only = np.sum(~x & y)
            both_wrong = np.sum(~x & ~y)

            table = [
                [both_correct, a_only],
                [b_only, both_wrong]
            ]

            result = mcnemar(
                table,
                exact=True
            )

            row[f"@{thr}"] = p_to_str(result.pvalue)

        rows.append(row)

    df = pd.DataFrame(rows)

    print("=" * 100)
    print(f"McNemar Test — Viewpoint Extremeness {vp + 1}")
    print("=" * 100)

    display(df)

Homography Correctness Mcnemar test by viewpoint sl
McNemar Test — Viewpoint Extremeness 1


,Comparison,@1,@3,@5,@10,@20,@50
0,Orig-ISP vs VO-ISP,0.219,1,1,1,1,1
1,Orig-ISP vs FO-ISP (Proxy),0.0625,1,1,1,1,1
2,Orig-ISP vs FO-ISP (CMA-ES),0.0625,1,1,1,1,1
3,VO-ISP vs FO-ISP (Proxy),1,1,1,1,1,1
4,VO-ISP vs FO-ISP (CMA-ES),1,1,1,1,1,1
5,FO-ISP (Proxy) vs FO-ISP (CMA-ES),1,1,1,1,1,1


McNemar Test — Viewpoint Extremeness 2


,Comparison,@1,@3,@5,@10,@20,@50
0,Orig-ISP vs VO-ISP,1,1,1,1,1,1
1,Orig-ISP vs FO-ISP (Proxy),0.508,1,1,1,1,1
2,Orig-ISP vs FO-ISP (CMA-ES),0.549,1,1,1,1,1
3,VO-ISP vs FO-ISP (Proxy),0.508,1,1,1,1,1
4,VO-ISP vs FO-ISP (CMA-ES),0.508,1,1,1,1,1
5,FO-ISP (Proxy) vs FO-ISP (CMA-ES),1,1,1,1,1,1


McNemar Test — Viewpoint Extremeness 3


,Comparison,@1,@3,@5,@10,@20,@50
0,Orig-ISP vs VO-ISP,0.5,0.774,1,1,1,1
1,Orig-ISP vs FO-ISP (Proxy),1,1,1,1,1,1
2,Orig-ISP vs FO-ISP (CMA-ES),1,1,0.625,1,1,1
3,VO-ISP vs FO-ISP (Proxy),0.375,0.549,0.774,1,1,1
4,VO-ISP vs FO-ISP (CMA-ES),1,0.453,0.549,1,1,1
5,FO-ISP (Proxy) vs FO-ISP (CMA-ES),0.688,1,1,1,1,1


McNemar Test — Viewpoint Extremeness 4


,Comparison,@1,@3,@5,@10,@20,@50
0,Orig-ISP vs VO-ISP,1,1,0.125,0.688,0.5,1
1,Orig-ISP vs FO-ISP (Proxy),1,0.289,0.125,0.219,1,1
2,Orig-ISP vs FO-ISP (CMA-ES),1,1,0.754,1,0.625,1
3,VO-ISP vs FO-ISP (Proxy),1,0.549,1,0.688,1,1
4,VO-ISP vs FO-ISP (CMA-ES),1,1,0.508,0.688,0.125,1
5,FO-ISP (Proxy) vs FO-ISP (CMA-ES),1,0.289,0.581,0.219,0.375,1


McNemar Test — Viewpoint Extremeness 5


,Comparison,@1,@3,@5,@10,@20,@50
0,Orig-ISP vs VO-ISP,1,1,1,0.5,0.625,1
1,Orig-ISP vs FO-ISP (Proxy),1,1,1,1,0.453,0.125
2,Orig-ISP vs FO-ISP (CMA-ES),1,1,0.625,1,0.375,0.453
3,VO-ISP vs FO-ISP (Proxy),1,1,1,0.25,1,0.125
4,VO-ISP vs FO-ISP (CMA-ES),1,1,0.625,0.375,1,0.688
5,FO-ISP (Proxy) vs FO-ISP (CMA-ES),1,1,1,1,1,0.688
